# Reproducibility Notebook

This notebook reproduces the main saved results for the rigorous echoed-ansatz follow-up study. It loads the machine-readable outputs, rebuilds the key summary tables from the saved CSV, inspects a representative case artifact, and optionally reruns the full production study if you enable the guarded flag in the parameter cell.

## Tunable Parameters

Edit the variables in the next cell to change the focus case, choose a construction to inspect, or enable the expensive full rerun path. The default behavior is to load the saved study outputs.

In [ ]:
from pathlib import Path

STUDY_DIR = Path.cwd().resolve().parent
DATA_DIR = STUDY_DIR / 'data'
ARTIFACT_CASE_DIR = STUDY_DIR / 'artifacts' / 'cases'
FIGURES_DIR = STUDY_DIR / 'figures'

FOCUS_CASE_ID = 'chi_plus_chiprime_structured_xy_na2_chiT5p0'
FOCUS_CONSTRUCTION = 'echo_opt_ideal'
RERUN_FULL_STUDY = False
RERUN_ARGS = [
    '--direct-starts', '2',
    '--direct-maxiter', '10',
    '--segment-maxiter', '8',
    '--echo-ideal-maxiter', '8',
    '--echo-gaussian-maxiter', '5',
    '--refocus-maxiter', '8',
]

print(STUDY_DIR)

In [ ]:
import json
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display

summary = json.loads((DATA_DIR / 'study_summary.json').read_text(encoding='utf-8'))
validation = json.loads((DATA_DIR / 'validation_summary.json').read_text(encoding='utf-8'))
metric_defs = json.loads((DATA_DIR / 'metric_definitions.json').read_text(encoding='utf-8'))
df = pd.read_csv(DATA_DIR / 'study_results.csv')

display(Markdown('Loaded **{}** construction rows across **{}** study cases.'.format(len(df), df['case_id'].nunique())))
summary

In [ ]:
display(Markdown('## Metric Definitions'))
pd.DataFrame(metric_defs['metric_definitions'].items(), columns=['metric', 'meaning'])

In [ ]:
display(Markdown('## Construction Summary'))
pd.DataFrame(summary['construction_summary'])

In [ ]:
display(Markdown('## Direct Versus Echo Comparison'))
pivot = df[df.construction.isin(['full_shared_line', 'full_shared_line_total_matched', 'echo_replay_ideal', 'echo_opt_ideal', 'echo_opt_gaussian', 'echo_opt_multitone_pi'])][['case_id', 'construction', 'restricted_average_gate_fidelity', 'max_residual_z_error_rad', 'probe_fidelity_mean']]
pivot.pivot(index='case_id', columns='construction', values='restricted_average_gate_fidelity').round(6)

In [ ]:
focus_path = ARTIFACT_CASE_DIR / ('{}_{}.json'.format(FOCUS_CASE_ID, FOCUS_CONSTRUCTION))
focus = json.loads(focus_path.read_text(encoding='utf-8'))
display(Markdown('## Focus Artifact: `{}`'.format(focus_path.name)))
focus['summary_row']

In [ ]:
probe_rows = pd.DataFrame([{'label': row['label'], 'state_fidelity': row['state_fidelity']} for row in focus['probe_metrics']['probe_rows']])
display(Markdown('### Focus Probe-State Fidelities'))
probe_rows

In [ ]:
display(Markdown('## Validation Payload'))
validation

In [ ]:
if RERUN_FULL_STUDY:
    cmd = [sys.executable, str(STUDY_DIR / 'scripts' / 'run_study.py'), *RERUN_ARGS]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=STUDY_DIR.parent.parent)
else:
    print('RERUN_FULL_STUDY is False. Set it to True in the parameter cell if you want to recompute the full study.')